# DecodeLabs Industrial Training Kit
## Project 2: Data Classification Using AI
### *From Raw Data to Intelligent Decision Making (Supervised Learning Pipeline)*

---
**Batch:** 2026 | **Author:** Shivam | **Powered by:** DecodeLabs  
**Track:** Artificial Intelligence Engineer  

> **Goal:** Build an end-to-end classification model using the Iris Benchmark dataset following the **IPO (Input, Process, Output)** architectural blueprint.

## 1. Architectural Paradigms: Heuristic vs Supervised Learning

Traditional programming relies on hardcoded heuristics:
```
IF condition A THEN action 1
ELSE IF condition B THEN action 2
```

In **Supervised Learning**, we provide historical data and class labels $(X, y)$, and the machine learns mathematical decision boundaries directly from data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Libraries imported successfully!")

## 2. [PHASE 1: INPUT] The Iris Benchmark Dataset

The dataset consists of:
- **Samples:** 150 total records
- **Classes (3 balanced):** Setosa, Versicolor, Virginica (50 samples each)
- **Dimensions (4 features):**
  1. `sepal length (cm)`
  2. `sepal width (cm)`
  3. `petal length (cm)`
  4. `petal width (cm)`

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species_name'] = df['target'].map(lambda i: iris.target_names[i])

print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df['species_name'].value_counts())

df.head()

In [ ]:
# Statistical Summary
df.describe().T

In [ ]:
# Exploratory Pairplot
sns.pairplot(df, hue='species_name', palette=['#2563eb', '#16a34a', '#dc2626'], corner=True)
plt.suptitle('Iris Feature Distributions by Species', y=1.02, fontsize=14, weight='bold')
plt.show()

## 3. [PHASE 2: PROCESS] Feature Scaling & Train-Test Split

### The Gatekeeper Rule: Scaling
K-Nearest Neighbors computes Euclidean distances:  
$$d(p, q) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}$$

If one feature has a scale of $0 - 1000$ and another is $0 - 1$, the larger feature dominates distance computations.  
Using `StandardScaler`, we normalize each feature so $\mu = 0$ and $\sigma^2 = 1$:
$$z = \frac{x - \mu}{\sigma}$$

### Structural Integrity: The Split
- 80% Training Set (Pattern Recognition)
- 20% Test Set (Validation / Generalization check)
- Stratified shuffle to preserve equal class proportions.

In [ ]:
X = df[iris.feature_names]
y = df['target']

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y, shuffle=True
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Samples: {X_train_scaled.shape[0]}")
print(f"Testing Samples:  {X_test_scaled.shape[0]}")
print(f"Scaled Feature Mean (Train): {np.mean(X_train_scaled, axis=0).round(4)}")
print(f"Scaled Feature Std  (Train): {np.std(X_train_scaled, axis=0).round(4)}")

## 4. The Algorithm: K-Nearest Neighbors (KNN)

**The Proximity Principle:** Similar things exist in close proximity.  
Given a query point $x$, KNN finds the $K$ closest training samples and assigns the class via **majority vote**.

### Tuning the Engine: Choosing "K" (The Elbow Method)
- $K=1$: Prone to noise & overfitting
- $K$ too large: Generic & underfitting
- **Optimal $K$ (The Elbow):** Balances variance and bias.

In [ ]:
k_range = range(1, 21)
test_errors = []
train_errors = []
test_accuracies = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_err = np.mean(knn.predict(X_train_scaled) != y_train)
    test_err = np.mean(knn.predict(X_test_scaled) != y_test)
    acc = accuracy_score(y_test, knn.predict(X_test_scaled))
    
    train_errors.append(train_err)
    test_errors.append(test_err)
    test_accuracies.append(acc)

optimal_k = list(k_range)[np.argmin(test_errors)]
print(f"Optimal K determined: {optimal_k} (Test Error Rate: {min(test_errors):.4f})")

# Plot Elbow Curve
plt.figure(figsize=(9, 5))
plt.plot(k_range, test_errors, marker='o', color='#2563eb', label='Test Error Rate', lw=2)
plt.plot(k_range, train_errors, marker='s', linestyle='--', color='#94a3b8', label='Train Error Rate')
plt.plot(optimal_k, min(test_errors), marker='*', markersize=16, color='#dc2626', label=f'Optimal K = {optimal_k}')
plt.title('Hyperparameter Tuning: Choosing K via Elbow Method', fontsize=13, weight='bold')
plt.xlabel('K (Number of Nearest Neighbors)')
plt.ylabel('Error Rate')
plt.xticks(k_range)
plt.legend()
plt.show()

## 5. Model Training: Scikit-Learn 3-Step Workflow

1. **Instantiate:** `model = KNeighborsClassifier(n_neighbors=optimal_k)`
2. **Fit:** `model.fit(X_train_scaled, y_train)`
3. **Predict:** `predictions = model.predict(X_test_scaled)`

In [ ]:
# Train Final KNN Model
final_knn = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn.fit(X_train_scaled, y_train)
y_pred = final_knn.predict(X_test_scaled)

print(f"KNN (K={optimal_k}) Model Trained Successfully!")

## 6. [PHASE 3: OUTPUT & DIAGNOSTICS] Diagnostic Tools & Metrics

### Beyond Accuracy: Strategic Trade-Offs
- **Confusion Matrix:** Measures True Positives ($TP$), False Positives ($FP$), False Negatives ($FN$), and True Negatives ($TN$).
- **Precision:** $\frac{TP}{TP + FP}$ (Trustworthiness / False Alarm avoidance)
- **Recall / Sensitivity:** $\frac{TP}{TP + FN}$ (Missed detection avoidance)
- **F1-Score:** Harmonic Mean of Precision & Recall:  
  $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Overall Test Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Plot Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title('DecodeLabs AI - Confusion Matrix Heatmap', fontsize=12, weight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual True Label')
plt.show()

## 7. Algorithm Benchmark Comparison
Comparing KNN with other supervised classifiers: Logistic Regression, Decision Trees, Random Forest, and Support Vector Machines.

In [ ]:
benchmark_models = {
    f"KNN (K={optimal_k})": KNeighborsClassifier(n_neighbors=optimal_k),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42),
    "Support Vector Classifier": SVC(probability=True, random_state=42)
}

results = []
for name, clf in benchmark_models.items():
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')
    results.append({"Algorithm": name, "Accuracy": acc * 100, "Weighted F1": f1 * 100})

df_benchmark = pd.DataFrame(results)
display(df_benchmark)

# Visual Comparison
df_benchmark.set_index('Algorithm').plot(kind='bar', figsize=(10, 5), color=['#3b82f6', '#10b981'])
plt.title('Model Benchmark: Supervised Learning Algorithms Comparison', weight='bold')
plt.ylabel('Score (%)')
plt.ylim(85, 102)
plt.xticks(rotation=20, ha='right')
plt.show()

## 8. Decision Boundary Visualizations
Visualizing 2D decision contours in feature space.

In [ ]:
from matplotlib.colors import ListedColormap

# Using Petal Length (index 2) and Petal Width (index 3)
X_petal_2d = X_train_scaled[:, [2, 3]]
clf_petal = KNeighborsClassifier(n_neighbors=optimal_k)
clf_petal.fit(X_petal_2d, y_train)

x_min, x_max = X_petal_2d[:, 0].min() - 1, X_petal_2d[:, 0].max() + 1
y_min, y_max = X_petal_2d[:, 1].min() - 1, X_petal_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

Z = clf_petal.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
cmap_light = ListedColormap(['#bfdbfe', '#bbf7d0', '#fef08a'])
colors = ['#1d4ed8', '#15803d', '#b45309']

plt.contourf(xx, yy, Z, alpha=0.5, cmap=cmap_light)
for idx, name in enumerate(iris.target_names):
    pts = X_petal_2d[y_train == idx]
    plt.scatter(pts[:, 0], pts[:, 1], color=colors[idx], edgecolor='k', s=50, label=name.capitalize())

plt.title(f'Decision Boundaries (Petal Length vs Petal Width, K={optimal_k})', weight='bold')
plt.xlabel('Petal Length (Standard Scaled)')
plt.ylabel('Petal Width (Standard Scaled)')
plt.legend()
plt.show()

## 9. [PHASE 4: LIVE INFERENCE] Testing Unseen Samples
Predicting flower species for newly observed measurements.

In [ ]:
def classify_flower(sepal_len, sepal_wid, petal_len, petal_wid):
    raw = np.array([[sepal_len, sepal_wid, petal_len, petal_wid]])
    scaled = scaler.transform(raw)
    pred_class = final_knn.predict(scaled)[0]
    probs = final_knn.predict_proba(scaled)[0]
    species = iris.target_names[pred_class]
    
    print(f"Measurements: Sepal({sepal_len}x{sepal_wid}cm), Petal({petal_len}x{petal_wid}cm)")
    print(f"-> Predicted Species: {species.upper()} (Confidence: {probs[pred_class]*100:.1f}%)")
    for i, name in enumerate(iris.target_names):
        print(f"   * {name:<10}: {probs[i]*100:.1f}%")
    return species

# Test on 3 diverse sample types
print("Sample 1:")
classify_flower(5.0, 3.4, 1.5, 0.2)
print("\nSample 2:")
classify_flower(6.0, 2.8, 4.5, 1.4)
print("\nSample 3:")
classify_flower(6.7, 3.3, 5.7, 2.3)

## 10. Conclusion & Emerging Horizons

We have successfully built, validated, and tuned a complete AI Supervised Classification pipeline for DecodeLabs Project 2.

- **Mastered Fundamentals:** Feature scaling, stratified train-test splits, distance-based classification, and hyperparameter tuning with the Elbow method.
- **Beyond Accuracy:** Analyzed diagnostic metrics (Precision, Recall, F1 harmonic mean, and Confusion Matrix).
- **Emerging Horizons:** From tabular data classification $\rightarrow$ Computer Vision, Deep Learning & Convolutional Neural Networks (CNNs) in subsequent modules.